# Supplementary Table 7: Mann–Whitney U Tests for ARI Comparisons


In [13]:
import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

PATH = '../results/selected_baseline_ari_runs.csv'
runs = pd.read_csv(PATH, encoding="utf-8-sig", float_precision="round_trip")

In [14]:
records = []
for dataset in ["Lymph Node A1", "Lymph Node D1", "E18.5", "Human Melanoma"]:
    group = runs[runs["Dataset"] == dataset]
    spa = group.loc[group["Method"] == "SpaGenFuse", "ARI"].to_numpy()
    baseline_rows = group[group["Method"] != "SpaGenFuse"]
    baseline = baseline_rows["ARI"].to_numpy()
    test = mannwhitneyu(spa, baseline, alternative="two-sided", method="exact")
    records.append({
        "Dataset": dataset,
        "Strongest baseline": baseline_rows["Method"].iloc[0],
        "SpaGenFuse ARI (mean ± SD)": f"{spa.mean():.4f} ± {spa.std(ddof=1):.4f}",
        "Baseline ARI (mean ± SD)": f"{baseline.mean():.4f} ± {baseline.std(ddof=1):.4f}",
        "Mann–Whitney U": int(test.statistic),
        "Raw p value": test.pvalue,
    })

table = pd.DataFrame(records)
p_values = table["Raw p value"].to_numpy()
order = np.argsort(p_values)
adjusted = np.empty(len(p_values))
adjusted[order] = np.minimum(1, np.maximum.accumulate(
    (len(p_values) - np.arange(len(p_values))) * p_values[order]
))
table["Holm-adjusted p value"] = adjusted
table["Strongest baseline"] = table["Strongest baseline"].replace({"TotalVI": "totalVI"})
table.round(6)

,Dataset,Strongest baseline,SpaGenFuse ARI (mean ± SD),Baseline ARI (mean ± SD),Mann–Whitney U,Raw p value,Holm-adjusted p value
0,Lymph Node A1,PRAGA,0.3285 ± 0.0054,0.2562 ± 0.0159,25,0.007937,0.031746
1,Lymph Node D1,totalVI,0.2928 ± 0.0141,0.1918 ± 0.0191,25,0.007937,0.031746
2,E18.5,SpatialGlue,0.6173 ± 0.0041,0.5179 ± 0.0735,25,0.007937,0.031746
3,Human Melanoma,GraphST,0.6671 ± 0.0008,0.5844 ± 0.0649,20,0.150794,0.150794
